In [5]:
import cv2


In [6]:
# FIRST HAVE TO CAPTURE THE VIDEO AND RENDER FRAMES
capture = cv2.VideoCapture("volleyball.mp4")

if not capture.isOpened():
   raise RuntimeError("Error: Could not open video.")

else:
    print("Video opened successfully")

fps = capture.get(cv2.CAP_PROP_FPS) #returns the frame rate of the video
frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT)) # total number of frames in the video
frame_index = 0

for frame_index in range(frame_count):

    success, frame = capture.read() # Success is a boolean variable

    if success:
        # Display the frame using imshow
        cv2.imshow("Ball tracking", frame)
        key_press = cv2.waitKey(0) & 0xFF  # Wait for a key press
        if key_press == ord("q"):
            break  # Exit the loop if 'q' is pressed
        if key_press == ord("s"):
            cv2.imwrite("sample_frame.png", frame)  # Save the current frame as an image

    else:
        print("Error: Could not read the frame.")
        capture.release()  # Release the video capture object
        break

    height, width = frame.shape[:2]

    if frame_index == 0:
        print(height, width)  # Print the shape of the first frame

capture.release()  # Release the video capture object
cv2.destroyAllWindows()  # Close all OpenCV windows



Video opened successfully
1080 1920


In [7]:
#KEEP RE RUNNING THIS CELL TO RESELECT THE BALL PATCH UNTIL YOU ARE SATISFIED WITH THE SELECTION

sample = cv2.imread("sample_frame.png")  # Read the saved sample frame
if sample is None:
    raise RuntimeError("Error: Could not read the sample frame.")

hsv_sample = cv2.cvtColor(sample, cv2.COLOR_BGR2HSV)  # Convert the sample frame to HSV color space

x, y, width, height = cv2.selectROI("Select ball patch", sample)
# cv2.destroyWindow("Select ball patch")  # Close the ROI selection window

ball_patch = hsv_sample[y:y + height, x:x + width]  # Extract the selected ball patch

pixels = ball_patch.reshape(-1, 3)  # Reshape the ball patch to a list of pixels
min_hsv_vals = pixels.min(axis=0)  # Get the minimum H, S, V values
max_hsv_vals = pixels.max(axis=0)  # Get the maximum H, S, V values
print("Minimum H, S, V:", min_hsv_vals)  # Print the minimum H, S, V values
print("Maximum H, S, V:", max_hsv_vals)  # Print the maximum H, S, V values


Minimum H, S, V: [18  4 56]
Maximum H, S, V: [165 165 189]


In [8]:
#KEEP CHANGING THE HSV RANGE UNTIL YOU ARE SATISFIED WITH THE MASK
# Create a mask for the ball using the selected patch
mask = cv2.inRange(hsv_sample, (22, 107, 120), (30, 177, 216))  # Adjust the HSV range based on the selected patch (these values are finalised tuned values)

cv2.imshow("Ball Mask", mask)  # Display the ball mask
cv2.waitKey(0)  # Wait for a key press
cv2.destroyAllWindows()  # Close all OpenCV windows

contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)  # Find contours in the mask

preview = sample.copy()  # Create a copy of the sample frame for previewing


for contour in contours:
    area = cv2.contourArea(contour)  # Calculate the area of the contour
    if 450 < area < 550:  # Filter out small contours based on area
        print(area)  # Print the area of the contour
        cv2.drawContours(preview, [contour], -1, (0, 255, 0), 2)  # Draw the contour on the preview image
        x, y, w, h = cv2.boundingRect(contour)  # Get the bounding rectangle for each contour
        cv2.rectangle(preview, (x, y), (x + w, y + h), (255, 0, 0), 2)  # Draw the bounding rectangle
        cv2.putText(preview, f"{area:.0f}", (x, y - 5),
        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)



cv2.imshow("Candidates", preview)  # Display the preview with contours
cv2.waitKey(0)  # Wait for a key press
cv2.destroyAllWindows()  # Close all OpenCV windows